# Un millón de dólares contra el abandono escolar

**El problema.** Somos el equipo técnico del MINEDUC a fines de 2021. Hay que repartir US$ 1.000.000 entre los alumnos de la foto 2021 para evitar la mayor cantidad de abandonos en 2022. Invertir `m` dólares en un alumno baja su riesgo en `efecto(m) = 0,6 · (1 − e^(−m/200))`.

**Cómo leer este notebook.** Las celdas marcadas con 🟨 **DECISIÓN** son puntos donde decido yo antes de seguir. Antes de cada una, el notebook pone números sobre la mesa.

| Sección | Estado |
|---|---|
| 1. Datos y población | ✅ |
| 2. FE de la clase | ✅ |
| 3. FE nuevo (cinco grupos) | ✅ |
| 4. Prueba de fuga | ✅ |
| 5. Backtests: ¿qué suma cada grupo? | ✅ |
| 🟨 6. Decisión: qué features quedan | ✅ |
| 7. Modelos finales 2015–2020 | ✅ |
| 8. Escenarios de calibración | ✅ |
| 9. Reparto y matriz de arrepentimiento | ✅ |
| 🟨 10. Regla de elección y entrega | ✅ |

**Decisiones ya tomadas**
- Criterio: maximizar los abandonos evitados, sin restricciones adicionales.
- El escenario de 2022 se elige con los datos (sección 8), sin usar información externa.

## 1. Datos y población

Juntamos las fotos 2014–2020 (con etiqueta) y la foto 2021 (sin etiqueta) en una sola tabla. Es necesario: el FE de 2021 necesita la historia de cada alumno en 2020, 2019, etc.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DATOS = Path("../predicciones/clase6/desafio_ministerio")
SEMILLA = 102191
SIT_FIN = {"P": 0, "R": 1, "Y": 2}

d = pd.concat([pd.read_csv(DATOS / "chile_2014_2020.csv.gz"),
               pd.read_csv(DATOS / "chile_2021.csv.gz")], ignore_index=True)
d["sit_fin"] = d["sit_fin"].map(SIT_FIN)
d = d.sort_values(["id_alumno", "agno"]).reset_index(drop=True)

d.groupby("agno").agg(filas=("id_alumno", "size"), egresan=("egresa", "sum"),
                      tasa_abandono=("abandono_t1", "mean")).round(4)

,filas,egresan,tasa_abandono
agno,,,
2014,286682,18629,0.0143
2015,286747,19082,0.0143
2016,286994,19062,0.0134
2017,288367,18875,0.0139
2018,291259,19768,0.0146
2019,293716,19753,0.0183
2020,293882,20575,0.0114
2021,298173,20632,NaN


- 2021 no tiene tasa porque su etiqueta mira 2022, que es justamente lo que se evalúa.
- Los egresados no pueden abandonar: se sacan para modelar y en 2021 reciben monto 0. El FE se calcula **antes** de sacarlos, con la historia completa.

## 2. El FE de la clase

Son las 11 variables de `abandono_chile/python/fe.py`, reescritas acá para poder modificarlas. Todas miran hacia atrás (el mismo alumno en años anteriores) o al mismo año t.

Un cambio respecto del original: `repitencias_previas` se calcula con una suma acumulada en lugar de un `lambda` por alumno. Da lo mismo y es mucho más rápido.

In [2]:
CRUDAS = ["cod_reg_rbd", "cod_com_rbd", "cod_depe2", "rural_rbd", "cod_ense2", "cod_grado", "cod_jor",
          "cod_rama", "cod_espe", "gen_alu", "edad", "prom_gral", "asistencia", "sit_fin", "n_reg",
          "cambio_esc_anio", "esc_n", "esc_prom", "esc_asist", "esc_rep", "esc_ab_lag"]

FE_CLASE = ["prom_gral_lag1", "asistencia_lag1", "sit_fin_lag1", "prom_gral_delta1", "asistencia_delta1",
            "repitencias_previas", "anios_previos", "cambio_esc_entre", "sobreedad", "prom_vs_esc",
            "asist_vs_esc"]


def fe_clase(d):
    g = d.groupby("id_alumno", sort=False)
    seguido = g["agno"].shift(1) == d["agno"] - 1
    for v in ["prom_gral", "asistencia", "sit_fin"]:
        d[f"{v}_lag1"] = g[v].shift(1).where(seguido)
    for v in ["prom_gral", "asistencia"]:
        d[f"{v}_delta1"] = d[v] - d[f"{v}_lag1"]
    rep = (d["sit_fin"] == 1).astype(int)
    d["repitencias_previas"] = rep.groupby(d["id_alumno"]).cumsum() - rep
    d["anios_previos"] = g.cumcount()
    d["cambio_esc_entre"] = (g["id_escuela"].shift(1) != d["id_escuela"]).astype(float).where(seguido)
    d["sobreedad"] = d["edad"] - d.groupby(["agno", "cod_ense2", "cod_grado"])["edad"].transform("median")
    d["prom_vs_esc"] = d["prom_gral"] - d["esc_prom"]
    d["asist_vs_esc"] = d["asistencia"] - d["esc_asist"]
    return d

## 3. FE nuevo: cinco grupos, cada uno con su porqué

Cada grupo sale de algo que aprendimos en la clase. Se pueden prender y apagar por separado, así en la sección 5 se mide qué suma cada uno.

| Grupo | Qué hace | De dónde sale la idea |
|---|---|---|
| **A · Retirados sin ceros falsos** | Para `sit_fin = Y`, promedio y asistencia pasan de 0 a NaN *antes* de calcular lags y deltas | El 0 del retirado es "sin dato": ensucia `asistencia_delta1` (el −77) y el lag del año siguiente |
| **B · Huecos y abandonos previos** | `abandonos_previos`, `huecos_previos`, `anios_desde_previo` | En 2019, nada del FE le decía al modelo "este chico ya dejó la escuela una vez" |
| **C · Relativas al año** | Percentil de asistencia y promedio dentro del año, y dentro del año y grado | En 2020 la asistencia cambió de nivel (remoto). El percentil mira la posición, no el valor |
| **D · Escuela en movimiento** | Cambio de asistencia, promedio y tamaño de la escuela respecto del año anterior | Separar "mi escuela cerró o se achicó" de "yo falté" (estallido, pandemia) |
| **E · Trayectoria larga** | Asistencia y promedio de t−2 | Un solo año de historia puede ser ruido; dos años muestran tendencia |

Y una que se **saca**: `anios_previos`, que deriva con el calendario (train 1–2, test 4). Se evalúa con y sin ella.

### Sobre la fuga en B
`abandonos_previos` usa `abandono_t1` de **filas anteriores** del mismo alumno. El de la fila t−1 se conoce en t (es si estaba en t), así que en el año t ya está disponible. El de la propia fila t no se usa nunca: por eso se resta en el código. En las filas de 2021 los abandonos de 2020 hacia atrás ya se conocen.

In [3]:
def fe_nuevo(d, A=False, B=False, C=False, D=False, E=False):
    d = d.copy()
    if A:
        d.loc[d["sit_fin"] == 2, ["prom_gral", "asistencia"]] = np.nan
    d = fe_clase(d)
    g = d.groupby("id_alumno", sort=False)
    previo = g["agno"].shift(1)
    nuevas = []

    if B:
        ab = d["abandono_t1"].fillna(0)
        d["abandonos_previos"] = ab.groupby(d["id_alumno"]).cumsum() - ab
        hueco = ((d["agno"] - previo) > 1).astype(int)
        d["huecos_previos"] = hueco.groupby(d["id_alumno"]).cumsum()
        d["anios_desde_previo"] = d["agno"] - previo
        nuevas += ["abandonos_previos", "huecos_previos", "anios_desde_previo"]

    if C:
        for v in ["asistencia", "prom_gral"]:
            d[f"{v}_pct_anio"] = d.groupby("agno")[v].rank(pct=True)
            d[f"{v}_pct_grado"] = d.groupby(["agno", "cod_ense2", "cod_grado"])[v].rank(pct=True)
            nuevas += [f"{v}_pct_anio", f"{v}_pct_grado"]

    if D:
        e = (d.groupby(["id_escuela", "agno"], as_index=False)[["esc_asist", "esc_prom", "esc_n"]].first()
              .sort_values(["id_escuela", "agno"]))
        ge = e.groupby("id_escuela")
        sig = ge["agno"].shift(1) == e["agno"] - 1
        e["esc_asist_delta1"] = (e["esc_asist"] - ge["esc_asist"].shift(1)).where(sig)
        e["esc_prom_delta1"] = (e["esc_prom"] - ge["esc_prom"].shift(1)).where(sig)
        e["esc_n_var1"] = (e["esc_n"] / ge["esc_n"].shift(1) - 1).where(sig)
        cols = ["esc_asist_delta1", "esc_prom_delta1", "esc_n_var1"]
        d = d.merge(e[["id_escuela", "agno"] + cols], on=["id_escuela", "agno"], how="left").set_index(d.index)
        nuevas += cols

    if E:
        seguido2 = (g["agno"].shift(2) == d["agno"] - 2) & (previo == d["agno"] - 1)
        for v in ["asistencia", "prom_gral"]:
            d[f"{v}_lag2"] = g[v].shift(2).where(seguido2)
            nuevas.append(f"{v}_lag2")

    return d, nuevas

In [4]:
d_raw, nuevas_raw = fe_nuevo(d, A=False, B=True, C=True, D=True, E=True)
d_A, nuevas_A = fe_nuevo(d, A=True, B=True, C=True, D=True, E=True)

GRUPOS = {"B": ["abandonos_previos", "huecos_previos", "anios_desde_previo"],
          "C": ["asistencia_pct_anio", "asistencia_pct_grado", "prom_gral_pct_anio", "prom_gral_pct_grado"],
          "D": ["esc_asist_delta1", "esc_prom_delta1", "esc_n_var1"],
          "E": ["asistencia_lag2", "prom_gral_lag2"]}
assert sorted(sum(GRUPOS.values(), [])) == sorted(nuevas_raw)
len(d_raw), len(nuevas_raw)

(2325820, 12)

### Un vistazo: el alumno del hueco

Es el mismo tipo de caso que vimos en el bloque 2. Mirá qué ve ahora el modelo en la fila posterior al hueco: `abandonos_previos` y `huecos_previos` dicen lo que antes había que adivinar a partir de los NaN.

In [5]:
ejemplo = d_A[(d_A["huecos_previos"] > 0) & (d_A["abandonos_previos"] > 0) & (d_A["agno"] == 2019)]["id_alumno"].iloc[0]
cols_ver = ["agno", "id_escuela", "cod_grado", "edad", "prom_gral", "asistencia", "sit_fin", "abandono_t1",
            "asistencia_lag1", "asistencia_delta1", "abandonos_previos", "huecos_previos",
            "asistencia_pct_anio", "esc_asist_delta1", "asistencia_lag2"]
d_A.loc[d_A["id_alumno"] == ejemplo, cols_ver].round(2)

,agno,id_escuela,cod_grado,edad,prom_gral,asistencia,sit_fin,abandono_t1,asistencia_lag1,asistencia_delta1,abandonos_previos,huecos_previos,asistencia_pct_anio,esc_asist_delta1,asistencia_lag2
738,2014,E05841,3,8,6.1,86.0,0.0,0.0,NaN,NaN,0.0,0,0.17,NaN,NaN
739,2015,E04380,4,9,6.4,81.0,0.0,1.0,86.0,-5.0,0.0,0,0.08,-0.30,NaN
740,2018,E07750,7,12,5.2,94.0,0.0,0.0,NaN,NaN,1.0,1,0.50,-1.06,NaN
741,2019,E05604,8,13,5.9,98.0,0.0,0.0,94.0,4.0,1.0,1,0.83,-1.61,NaN
742,2020,E03911,1,14,5.1,100.0,0.0,0.0,98.0,2.0,1.0,1,0.77,24.65,94.0
743,2021,E03911,2,15,4.4,9.0,1.0,NaN,100.0,-91.0,1.0,1,0.01,-92.74,98.0


## 4. Prueba de fuga

Es la prueba más directa: si calculo el FE **cortando los datos en 2018** y lo comparo con el FE calculado con **todos** los años, las filas de 2018 tienen que dar idénticas. Si alguna variable cambia, es que mira el futuro.

Ojo con lo que **no** prueba: que `abandonos_previos` use etiquetas pasadas está bien porque ya se conocen en t. Esta prueba detecta futuro, no uso de etiquetas.

In [6]:
cols_todas = FE_CLASE + nuevas_A
cortado, _ = fe_nuevo(d[d["agno"] <= 2018], A=True, B=True, C=True, D=True, E=True)
fa = cortado[cortado["agno"] == 2018].set_index("id_alumno")[cols_todas].sort_index()
fb = d_A[d_A["agno"] == 2018].set_index("id_alumno")[cols_todas].sort_index()
pd.testing.assert_frame_equal(fa, fb)
"sin fuga: las filas de 2018 no cambian al agregar 2019–2021"

'sin fuga: las filas de 2018 no cambian al agregar 2019–2021'

## 5. Backtests: ¿qué suma cada grupo?

Dos backtests, con la lógica de la clase:

| Backtest | Entrena | Evalúa | Para qué |
|---|---|---|---|
| **normal** | 2015–2017 | 2018 | El año "típico" de la clase |
| **pandemia** | 2015–2019 | 2020 | El año con etiqueta más parecido al que hay que predecir (2021 → 2022) |

**Hiperparámetros fijos:** los que eligió la búsqueda bayesiana en mi corrida de la clase. La cantidad de árboles se escala con las filas, como en `modelo_final`. No se vuelve a tunear: la búsqueda sumó 0,005 de AUC y el FE 0,046, así que el esfuerzo va primero al FE.

**Dos métricas:**
- **AUC**: qué tan bien ordena a todos.
- **Captura top 5 %**: de todos los que abandonan, qué fracción queda dentro del 5 % de alumnos más riesgosos según el modelo. Es la zona donde va a ir la plata.

In [7]:
PARAMS = {"objective": "binary", "verbose": -1, "seed": SEMILLA, "learning_rate": 0.03295,
          "num_leaves": 66, "min_data_in_leaf": 811, "feature_fraction": 0.6293, "lambda_l2": 4.786,
          "num_threads": 0}
ARBOLES_POR_FILA = 226 / 535_597
BACKTESTS = {"normal 2018": (2015, 2017, 2018), "pandemia 2020": (2015, 2019, 2020)}


def poblacion(dd):
    return dd[dd["egresa"] == 0]


def entrenar(dd, cols, desde, hasta):
    p = poblacion(dd)
    tr = p[p["agno"].between(desde, hasta)]
    n = max(1, round(ARBOLES_POR_FILA * len(tr)))
    return lgb.train(PARAMS, lgb.Dataset(tr[cols], tr["abandono_t1"]), n)


def captura_top(y, s, frac=0.05):
    top = np.argsort(-s)[: int(len(s) * frac)]
    return y[top].sum() / y.sum()


def evaluar(dd, cols):
    fila = {}
    for nombre, (desde, hasta, test) in BACKTESTS.items():
        m = entrenar(dd, cols, desde, hasta)
        te = poblacion(dd)
        te = te[te["agno"] == test]
        y, s = te["abandono_t1"].to_numpy(), m.predict(te[cols])
        fila[f"AUC {nombre}"] = roc_auc_score(y, s)
        fila[f"top5 {nombre}"] = captura_top(y, s)
    return fila

In [8]:
BASE = CRUDAS + FE_CLASE
SIN_AP = [c for c in BASE if c != "anios_previos"]
TODO = BASE + sum(GRUPOS.values(), [])

configs = {
    "clase (base)": (d_raw, BASE),
    "base − anios_previos": (d_raw, SIN_AP),
    "base + A retirados NaN": (d_A, BASE),
    "base + B huecos": (d_raw, BASE + GRUPOS["B"]),
    "base + C relativas": (d_raw, BASE + GRUPOS["C"]),
    "base + D escuela": (d_raw, BASE + GRUPOS["D"]),
    "base + E lag2": (d_raw, BASE + GRUPOS["E"]),
    "todo (A–E)": (d_A, TODO),
    "todo − anios_previos": (d_A, [c for c in TODO if c != "anios_previos"]),
}

filas = []
for nombre, (dd, cols) in configs.items():
    filas.append({"config": nombre, "n_vars": len(cols), **evaluar(dd, cols)})
    print(f"{nombre:28s} listo", flush=True)

ablacion = pd.DataFrame(filas).set_index("config")
ablacion

clase (base)                 listo


base − anios_previos         listo


base + A retirados NaN       listo


base + B huecos              listo


base + C relativas           listo


base + D escuela             listo


base + E lag2                listo


todo (A–E)                   listo


todo − anios_previos         listo


,n_vars,AUC normal 2018,top5 normal 2018,AUC pandemia 2020,top5 pandemia 2020
config,,,,,
clase (base),32,0.923752,0.719022,0.862970,0.531820
base − anios_previos,31,0.918028,0.709852,0.839783,0.492082
base + A retirados NaN,32,0.923466,0.717141,0.862727,0.534210
base + B huecos,35,0.923716,0.720903,0.862685,0.530624
base + C relativas,36,0.924146,0.720198,0.871697,0.540185
base + D escuela,35,0.923822,0.720903,0.861395,0.524649
base + E lag2,34,0.924268,0.719492,0.864114,0.529131
todo (A–E),44,0.924689,0.721608,0.869970,0.541679
todo − anios_previos,43,0.923414,0.721373,0.860242,0.526740


In [9]:
base = ablacion.loc["clase (base)"]
dif = (ablacion.drop(columns="n_vars") - base.drop("n_vars")).round(4)
dif.columns = [f"Δ {c}" for c in dif.columns]
dif

,Δ AUC normal 2018,Δ top5 normal 2018,Δ AUC pandemia 2020,Δ top5 pandemia 2020
config,,,,
clase (base),0.0000,0.0000,0.0000,0.0000
base − anios_previos,-0.0057,-0.0092,-0.0232,-0.0397
base + A retirados NaN,-0.0003,-0.0019,-0.0002,0.0024
base + B huecos,-0.0000,0.0019,-0.0003,-0.0012
base + C relativas,0.0004,0.0012,0.0087,0.0084
base + D escuela,0.0001,0.0019,-0.0016,-0.0072
base + E lag2,0.0005,0.0005,0.0011,-0.0027
todo (A–E),0.0009,0.0026,0.0070,0.0099
todo − anios_previos,-0.0003,0.0024,-0.0027,-0.0051


**Cómo leer las diferencias.** Un Δ AUC de +0,002 puede ser ruido. Como referencia, en la clase la búsqueda de hiperparámetros sumó +0,005 y el FE completo +0,046. Para tomar una decisión, conviene mirar si el signo se repite en los **dos** backtests y en las **dos** métricas.

### ¿Qué usa el modelo con todo?

Importancia por ganancia del modelo "todo (A–E)" en el backtest de pandemia. Las variables nuevas están marcadas. Recordatorio: **importancia no es efecto**, y variables que dicen lo mismo se reparten la importancia entre ellas.

In [10]:
m_todo = entrenar(d_A, TODO, 2015, 2019)
imp = pd.DataFrame({"variable": TODO, "ganancia": m_todo.feature_importance("gain")})
imp["ganancia"] = imp["ganancia"] / imp["ganancia"].sum()
imp["grupo"] = imp["variable"].map({v: g for g, vs in GRUPOS.items() for v in vs}).fillna(
    imp["variable"].map(lambda v: "FE clase" if v in FE_CLASE else "cruda"))
imp.sort_values("ganancia", ascending=False).head(20).round(4).reset_index(drop=True)

,variable,ganancia,grupo
0,prom_gral,0.1805,cruda
1,prom_gral_pct_anio,0.1255,C
2,asistencia_delta1,0.0784,FE clase
3,sit_fin,0.0697,cruda
4,asistencia_pct_anio,0.0621,C
5,asistencia,0.0599,cruda
6,esc_ab_lag,0.0583,cruda
7,edad,0.0452,cruda
8,cod_grado,0.0373,cruda
9,asist_vs_esc,0.0312,FE clase


## 🟨 6. DECISIÓN: qué features quedan ✅

Antes de decidir, dos evidencias que salieron de discutir la ablación.

**¿Por qué `anios_previos` importa tanto?** La deriva con el calendario hacía pensar que molestaba, pero sacarla resta mucho. Mirando por tipo de alumno se entiende: `anios_previos = 0` fuera de 1° básico delata a los que **entran al panel de golpe** (inmigrantes, chicos que habían dejado antes de 2014, que vienen de otras modalidades).

In [11]:
h = poblacion(d_raw)
h = h[h["agno"].between(2016, 2020)]
primero_basica = (h["cod_ense2"] == 2) & (h["cod_grado"] == 1)
h = h.assign(tipo=np.select(
    [(h["anios_previos"] == 0) & primero_basica, h["anios_previos"] == 0, h["asistencia_lag1"].isna()],
    ["nuevo en 1° básico", "nuevo en otro grado", "volvió tras un hueco"], "historia continua"))
h.groupby("tipo").agg(filas=("abandono_t1", "size"), tasa_abandono=("abandono_t1", "mean")).round(4)

,filas,tasa_abandono
tipo,,
historia continua,1213496,0.0120
nuevo en 1° básico,123075,0.0144
nuevo en otro grado,16528,0.2463
volvió tras un hueco,3086,0.1568


**¿Qué le pasa a D en la foto 2021?** Hay escuelas con asistencia media anómala, y sus alumnos reciben deltas de escuela que el modelo nunca vio al entrenar.

In [12]:
p_all = poblacion(d_raw)
pd.DataFrame({
    "% alumnos en escuelas con asistencia < 30": p_all.groupby("agno")["esc_asist"].apply(lambda s: 100 * (s < 30).mean()),
    "esc_asist_delta1 mínimo": p_all.groupby("agno")["esc_asist_delta1"].min(),
    "% con esc_asist_delta1 < −30": p_all.groupby("agno")["esc_asist_delta1"].apply(lambda s: 100 * (s < -30).mean()),
}).round(2)

,% alumnos en escuelas con asistencia < 30,esc_asist_delta1 mínimo,% con esc_asist_delta1 < −30
agno,,,
2014,0.06,NaN,0.00
2015,0.02,-92.50,0.01
2016,0.04,-90.73,0.03
2017,0.02,-100.00,0.01
2018,0.02,-100.00,0.00
2019,0.03,-95.42,0.05
2020,0.13,-98.48,0.38
2021,1.72,-99.02,3.54


### Decisiones

| | Decisión | Por qué |
|---|---|---|
| **A** · retirados NaN | ❌ | Δ ≈ 0 en los dos backtests y las dos métricas |
| **B** · huecos | ❌ | Δ ≈ 0. `anios_previos` > 0 con el lag vacío ya detecta el hueco del año anterior. Los huecos más viejos no, pero la ablación muestra que esa información no suma |
| **C** · percentiles | ✅ | Suma en pandemia (+0,009 AUC, +0,008 top 5 %) y no resta en el año normal. El percentil mira la **posición** dentro del año, así que aguanta cambios de nivel como los de 2020 y 2021 |
| **D** · escuela en movimiento | ❌ | ≈ 0 en el año normal y resta en pandemia. En 2021 hay deltas de escuela fuera de todo lo visto; los árboles no extrapolan y los mandan a la hoja más extrema del entrenamiento |
| **E** · lag2 | 🔀 se decide con sensibilidad | ≈ 0 en los backtests, pero en la foto 2021 `lag1` es 2020 (asistencia inflada) y `lag2` es 2019. **Ningún backtest reproduce esa situación**, así que la ablación no puede responderlo. Se entrenan dos modelos y en la sección 9 se mide si cambia el reparto |
| `anios_previos` | ✅ queda | Los nuevos fuera de 1° básico abandonan 20 veces más. Sacarla resta 0,023 de AUC en pandemia |

**Criterio:** pesa más el backtest de pandemia, porque 2021 se parece más a 2020 que a un año normal (validación adversarial con variables del alumno: AUC 0,686 contra 2020 y 0,840 contra 2018). Un grupo que da ≈ 0 no entra: suma complejidad, riesgo en 2021 y cosas que explicar.

## 7. Modelos finales: entrenan 2015–2020, predicen 2021

Dos variantes, que difieren solo en E. Entrenan con todas las fotos con etiqueta (2015–2020; 2014 no tiene historia previa) y predicen la foto 2021 sin egresados.

In [13]:
FINAL = BASE + GRUPOS["C"]
VARIANTES = {"sin E": FINAL, "con E": FINAL + GRUPOS["E"]}

foto = poblacion(d_raw)
foto = foto[foto["agno"] == 2021].copy()
modelos = {}
for nombre, cols in VARIANTES.items():
    modelos[nombre] = entrenar(d_raw, cols, 2015, 2020)
    foto[f"p {nombre}"] = modelos[nombre].predict(foto[cols])

foto[["p sin E", "p con E"]].describe(percentiles=[.5, .9, .99, .999]).T.round(4)

,count,mean,std,min,50%,90%,99%,99.9%,max
p sin E,277541.0,0.0123,0.0530,0.0001,0.0024,0.0154,0.2304,0.7633,0.9583
p con E,277541.0,0.0123,0.0528,0.0001,0.0024,0.0154,0.2352,0.7484,0.9585


In [14]:
def solapamiento(a, b, k):
    return len(set(np.argsort(-a)[:k]) & set(np.argsort(-b)[:k])) / k


pa, pb = foto["p sin E"].to_numpy(), foto["p con E"].to_numpy()
pd.DataFrame({
    "correlación de rangos": [pd.Series(pa).corr(pd.Series(pb), method="spearman")],
    "tasa media predicha sin E (%)": [100 * pa.mean()],
    "tasa media predicha con E (%)": [100 * pb.mean()],
    "comparten top 2.000": [solapamiento(pa, pb, 2000)],
    "comparten top 5.000": [solapamiento(pa, pb, 5000)],
}).round(4)

,correlación de rangos,tasa media predicha sin E (%),tasa media predicha con E (%),comparten top 2.000,comparten top 5.000
0,0.9622,1.2267,1.2251,0.916,0.9148


- **Tasa media predicha:** es la apuesta del modelo para 2022 si el año se comporta como el promedio de 2015–2020. Para comparar: 2014–2018 entre 1,44 % y 1,57 %; 2019, 1,96 %; 2020, 1,22 %.
- **Top 2.000 y 5.000:** con US$ 1.000.000, a US$ 500 o US$ 200 por chico, es aproximadamente a cuántos se llega. El solapamiento dice cuántos de esos chicos son los mismos con E y sin E. El efecto en el reparto se mide en la sección 9.

## 8. Escenarios de calibración

Sobre 2022 sabemos una sola cosa: los años no siempre se parecen al pasado. Hay dos "sorpresas" observadas con etiqueta, y se usan como **moldes** de lo que puede pasar:

| Escenario | Apuesta | Cómo se construye |
|---|---|---|
| **Normal** | 2022 se comporta como 2015–2018 | El modelo tal cual |
| **Como 2020** | Se repite la distorsión de la pandemia | Backtest 2015–2019 → 2020: se aprende el mapeo "predicho → real" (isotónica) y se aplica a las predicciones de 2021 |
| **Como 2019** | Se repite la sorpresa del estallido | Lo mismo, con el backtest 2015–2018 → 2019 |

**Lo que importa no es el nivel sino la forma.** Si todas las p se multiplican por el mismo factor, el reparto no cambia: el próximo dólar va al chico donde rinde más, y un factor común se cancela en esa comparación. Por eso el primer paso es ver **si la distorsión de cada año fue pareja**.

**Cómo leerlo:** la columna `real / predicho` de cada grupo de riesgo. Si es más o menos la misma en todos los grupos, la distorsión es pareja (solo nivel). Si cambia entre grupos, se deforma la forma y el reparto se mueve.

**Una aproximación a declarar:** el mapeo se aprende con un modelo de backtest (que entrena hasta 2018 o 2019) y se aplica al modelo final (que entrena hasta 2020). Se supone que la distorsión se parece entre los dos.

In [15]:
CORTES = [0, .005, .01, .02, .05, .1, .2, .4, 1]
CALIB = {"normal 2018": (2015, 2017, 2018), "estallido 2019": (2015, 2018, 2019), "pandemia 2020": (2015, 2019, 2020)}


def predecir_backtest(cols, desde, hasta, test):
    m = entrenar(d_raw, cols, desde, hasta)
    te = poblacion(d_raw)
    te = te[te["agno"] == test]
    return te["abandono_t1"].to_numpy(), m.predict(te[cols])


bt = {nombre: predecir_backtest(FINAL, *v) for nombre, v in CALIB.items()}

pd.DataFrame({nombre: {"tasa real (%)": 100 * y.mean(), "tasa predicha (%)": 100 * p.mean(),
                       "real / predicho": y.mean() / p.mean(), "AUC": roc_auc_score(y, p)}
              for nombre, (y, p) in bt.items()}).round(4)

,normal 2018,estallido 2019,pandemia 2020
tasa real (%),1.5665,1.9598,1.2246
tasa predicha (%),1.5222,1.6180,1.5720
real / predicho,1.0291,1.2112,0.7790
AUC,0.9241,0.9104,0.8717


In [16]:
def por_grupo(y, p):
    t = (pd.DataFrame({"y": y, "p": p, "grupo": pd.cut(p, CORTES, include_lowest=True)})
         .groupby("grupo", observed=True)
         .agg(alumnos=("y", "size"), abandonos=("y", "sum"), predicho=("p", "mean"), real=("y", "mean")))
    t["real / predicho"] = t["real"] / t["predicho"]
    return t


grupos = pd.concat({nombre: por_grupo(y, p) for nombre, (y, p) in bt.items()}, axis=1)
grupos.loc[:, (slice(None), ["predicho", "real"])] *= 100
grupos.round(3)

normal 2018                                            estallido 2019                                            pandemia 2020            \
                    alumnos abandonos predicho    real real / predicho        alumnos abandonos predicho    real real / predicho       alumnos abandonos   
grupo                                                                                                                                                      
(-0.001, 0.005]      211446     487.0    0.172   0.230           1.341         205112     678.0    0.178   0.331           1.852        162090     359.0   
(0.005, 0.01]         26934     263.0    0.691   0.976           1.413          31241     426.0    0.694   1.364           1.964         54787     367.0   
(0.01, 0.02]          12592     223.0    1.389   1.771           1.275          15770     390.0    1.384   2.473           1.786         28304     396.0   
(0.02, 0.05]           7877     271.0    3.074   3.440           1.119           8940     444.0    3.042   4.966           1.633         16114     472.0   
(0.05, 0.1]            3232     245.0    6.994   7.580           1.084           3015     336.0    6.924  11.144           1.609          4596     276.0   
(0.1, 0.2]             3131     490.0   14.720  15.650           1.063           3097     528.0   15.003  17.049           1.136          3175     283.0   
(0.2, 0.4]             4090    1039.0   28.681  25.403           0.886           4460    1045.0   27.593  23.430           0.849          2319     289.0   
(0.4, 1.0]             2189    1235.0   59.678  56.418           0.945           2328    1522.0   62.517  65.378           1.046          1922     905.0   

                                                  
                predicho    real real / predicho  
grupo                                             
(-0.001, 0.005]    0.247   0.221           0.896  
(0.005, 0.01]      0.701   0.670           0.955  
(0.01, 0.02]       1.381   1.399           1.013  
(0.02, 0.05]       3.045   2.929           0.962  
(0.05, 0.1]        6.855   6.005           0.876  
(0.1, 0.2]        14.606   8.913           0.610  
(0.2, 0.4]        27.294  12.462           0.457  
(0.4, 1.0]        63.372  47.086           0.743

- `predicho` y `real` están en %. `abandonos` es cuántos abandonos reales hay en el grupo: con pocos, el cociente es ruidoso.
- **Dónde mirar:** la plata va a llegar a unos 2.000–5.000 chicos, menos del 2 % de la foto. Eso cae en los grupos de riesgo alto (aproximadamente de 5–10 % para arriba). Una distorsión en los grupos bajos casi no mueve el reparto.

## 🟨 DECISIÓN: ¿la distorsión de 2019 es pareja? ✅

El reparto óptimo con el modelo tal cual le da plata solo a alumnos con riesgo predicho ≥ 11 % (sección 9). Así que importan las tres últimas filas:

| Grupo | Normal 2018 | Estallido 2019 | Pandemia 2020 |
|---|---|---|---|
| 10–20 % | 1,06 | 1,14 | 0,61 |
| 20–40 % | 0,89 | 0,85 | 0,46 |
| 40–100 % | 0,95 | 1,05 | 0,74 |

- **"Como 2019" se descarta.** En la zona de la plata, 2019 se comportó como un año normal. La sorpresa de 2019 (21 % más abandonos de los esperados) estuvo entre los chicos de **riesgo bajo**, que no reciben plata en ningún reparto: es un abandono invisible para este modelo.
- **"Como 2020" se queda.** La distorsión cae justo donde va la plata, y no es pareja: el grupo 20–40 % se achica más que los otros dos. Para el modelo, el grupo de arriba es 2,3 veces más riesgoso que el del medio; en el mundo 2020, 3,8 veces.
- **Quedan dos mundos:** normal y como 2020.
- **Con C en el modelo,** la distorsión de 2020 sigue: C mejora el orden (AUC 0,863 → 0,872) pero no la calibración.

### El mundo "como 2020": corrección isotónica

Se toma el backtest 2015–2019 → 2020 y se aprende una función creciente que lleva "lo que predijo el modelo" a "lo que pasó" (regresión isotónica). Después se aplica esa función a las predicciones de 2021.

- Es **creciente**, así que no cambia el orden de los alumnos: solo cambia las **proporciones** entre ellos, que es lo que mueve el reparto.
- Se aprende **una por variante** (con E y sin E), cada una con su propio backtest.

In [17]:
from sklearn.isotonic import IsotonicRegression

MUNDOS = ["normal", "como 2020"]
p = {}
for variante, cols in VARIANTES.items():
    y_bt, p_bt = bt["pandemia 2020"] if variante == "sin E" else predecir_backtest(cols, 2015, 2019, 2020)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-6).fit(p_bt, y_bt)
    p[(variante, "normal")] = foto[f"p {variante}"].to_numpy()
    p[(variante, "como 2020")] = iso.predict(p[(variante, "normal")])

pd.DataFrame({f"{v} · {w}": [100 * q.mean()] + [100 * q[(p[(v, 'normal')] > a) & (p[(v, 'normal')] <= b)].mean()
                                                 for a, b in [(.1, .2), (.2, .4), (.4, 1)]]
              for (v, w), q in p.items()},
             index=["tasa media 2021 (%)", "grupo 10–20 % (%)", "grupo 20–40 % (%)", "grupo 40–100 % (%)"]).round(2)

,sin E · normal,sin E · como 2020,con E · normal,con E · como 2020
tasa media 2021 (%),1.23,0.93,1.23,0.91
grupo 10–20 % (%),14.44,8.59,14.46,8.64
grupo 20–40 % (%),27.44,13.01,27.58,12.28
grupo 40–100 % (%),60.98,44.06,60.24,40.56


Los grupos se arman con la predicción del modelo tal cual. En el mundo "como 2020" cambian sus probabilidades, no quiénes son.

## 9. Reparto óptimo y matriz de arrepentimiento

### El reparto

Hay que maximizar los abandonos evitados esperados, Σ pᵢ · 0,6 · (1 − e^(−mᵢ/200)), gastando como mucho US$ 1.000.000.

**La idea es la del greedy:** cada dólar va al chico donde rinde más. El próximo dólar en el chico *i* rinde pᵢ · 0,003 · e^(−mᵢ/200), que baja a medida que el chico recibe plata. Repetido dólar por dólar, el algoritmo termina en un punto donde **el último dólar rinde lo mismo en todos los chicos que recibieron algo** (llamemos λ a ese rendimiento), y los que no recibieron nada no llegaban a λ ni con el primer dólar.

Despejando de "rinde λ", cada chico recibe **mᵢ = 200 · ln(pᵢ · 0,003 / λ)** si eso es positivo, y 0 si no. Es el mismo resultado que el greedy, sin el bucle de un millón de pasos. Solo falta buscar el λ que gasta exactamente el millón, y eso se hace por bisección.

Se ve en la fórmula lo que discutimos: si todas las p se multiplican por k, λ también se multiplica por k y los montos no cambian.

In [18]:
PRESUPUESTO = 1_000_000


def efecto(m):
    return 0.6 * (1 - np.exp(-m / 200))


def repartir(p, presupuesto=PRESUPUESTO):
    lo, hi = 1e-15, 1.0
    for _ in range(200):
        lam = np.sqrt(lo * hi)
        m = np.maximum(0, 200 * np.log(p * 0.003 / lam))
        lo, hi = (lam, hi) if m.sum() > presupuesto else (lo, lam)
    return m * presupuesto / m.sum()


def evitados(m, p):
    return (p * efecto(m)).sum()


repartos = {k: repartir(q) for k, q in p.items()}

pd.DataFrame({f"{v} · {w}": {"alumnos con plata": (m > 0).sum(),
                             "riesgo predicho mínimo con plata": p[(v, "normal")][m > 0].min(),
                             "monto máximo": m.max(),
                             "monto mediano (con plata)": np.median(m[m > 0]),
                             "abandonos evitados esperados (en su mundo)": evitados(m, p[(v, w)])}
              for (v, w), m in repartos.items()}).round(3)

,sin E · normal,sin E · como 2020,con E · normal,con E · como 2020
alumnos con plata,6419.000,8815.000,6399.000,8528.000
riesgo predicho mínimo con plata,0.111,0.066,0.112,0.068
monto máximo,431.875,545.366,430.221,547.653
monto mediano (con plata),124.070,67.207,127.213,63.436
abandonos evitados esperados (en su mundo),688.384,446.281,687.079,416.621


In [19]:
cortes_p = [0, .1, .2, .4, 1]
grupo = pd.cut(p[("sin E", "normal")], cortes_p, include_lowest=True)
pd.DataFrame({w: pd.Series(repartos[("sin E", w)]).groupby(grupo, observed=True).sum() for w in MUNDOS}).round(0)

,normal,como 2020
"(-0.001, 0.1]",0.0,38717.0
"(0.1, 0.2]",182009.0,192843.0
"(0.2, 0.4]",340533.0,249380.0
"(0.4, 1.0]",477459.0,519060.0


Cuánta plata va a cada grupo de riesgo (según el modelo tal cual), en el reparto pensado para cada mundo. Es la traducción a dólares de "en el mundo 2020 conviene darle más a los de arriba".

### La matriz de arrepentimiento

- **Filas:** los repartos que podríamos entregar.
- **Columnas:** el mundo que finalmente ocurre.
- **Valor:** abandonos evitados esperados del reparto de la fila, con las probabilidades del mundo de la columna.
- **Arrepentimiento:** lo mejor posible en esa columna menos la celda.

Se arma primero sin E (2 × 2) y después con las cuatro combinaciones (4 × 4), para ver si E importa.

In [20]:
def matriz(claves):
    valor = pd.DataFrame({f"mundo {v} · {w}": {f"reparto {rv} · {rw}": evitados(repartos[(rv, rw)], p[(v, w)])
                                              for rv, rw in claves}
                          for v, w in claves})
    return valor, valor.max() - valor


SIN_E = [("sin E", w) for w in MUNDOS]
valor, arrep = matriz(SIN_E)
display(valor.round(1))
arrep.assign(**{"peor caso": arrep.max(axis=1), "peor caso (%)": 100 * (arrep / valor.max()).max(axis=1)}).round(2)

,mundo sin E · normal,mundo sin E · como 2020
reparto sin E · normal,688.4,440.7
reparto sin E · como 2020,675.4,446.3


,mundo sin E · normal,mundo sin E · como 2020,peor caso,peor caso (%)
reparto sin E · normal,0.00,5.57,5.57,1.25
reparto sin E · como 2020,12.96,0.00,12.96,1.88


In [21]:
TODAS = list(p)
valor4, arrep4 = matriz(TODAS)
display(valor4.round(1))
arrep4.assign(**{"peor caso": arrep4.max(axis=1), "peor caso (%)": 100 * (arrep4 / valor4.max()).max(axis=1)}).round(2)

,mundo sin E · normal,mundo sin E · como 2020,mundo con E · normal,mundo con E · como 2020
reparto sin E · normal,688.4,440.7,679.6,408.3
reparto sin E · como 2020,675.4,446.3,666.2,411.7
reparto con E · normal,681.2,437.6,687.1,410.2
reparto con E · como 2020,667.2,440.8,671.4,416.6


,mundo sin E · normal,mundo sin E · como 2020,mundo con E · normal,mundo con E · como 2020,peor caso,peor caso (%)
reparto sin E · normal,0.00,5.57,7.50,8.30,8.30,1.99
reparto sin E · como 2020,12.96,0.00,20.90,4.91,20.90,3.04
reparto con E · normal,7.22,8.64,0.00,6.39,8.64,1.94
reparto con E · como 2020,21.16,5.44,15.68,0.00,21.16,3.07


In [22]:
def comparten(a, b):
    ra, rb = a > 0, b > 0
    return (ra & rb).sum() / (ra | rb).sum()


pd.DataFrame({f"{v} · {w}": {f"{v2} · {w2}": comparten(repartos[(v, w)], repartos[(v2, w2)]) for v2, w2 in TODAS}
              for v, w in TODAS}).round(3)

,sin E · normal,sin E · como 2020,con E · normal,con E · como 2020
sin E · normal,1.000,0.728,0.874,0.744
sin E · como 2020,0.728,1.000,0.720,0.880
con E · normal,0.874,0.720,1.000,0.750
con E · como 2020,0.744,0.880,0.750,1.000


**Solapamiento:** de todos los alumnos que reciben plata en alguno de los dos repartos, qué fracción la recibe en los dos. Un arrepentimiento bajo con solapamiento bajo diría que hay muchos repartos "casi óptimos".

## 🟨 10. DECISIÓN: regla de elección y entrega ✅

**Regla: minimax de arrepentimiento**, es decir, el reparto que menos pierde en el peor caso. El arrepentimiento promedio elige el mismo, en la matriz 2 × 2 y en la de 4 × 4.

**Elegido: "sin E · normal".** En el peor caso pierde 5,6 abandonos evitados esperados (1,3 %) si 2022 se comporta como 2020.

**E queda afuera, igual que A, B y D,** pero por otro camino. A, B y D se descartaron con la ablación. E no se podía evaluar así, porque ningún backtest reproduce la situación de 2021 (`lag1` = 2020 con asistencia inflada, `lag2` = 2019). Por eso se entrenaron las dos variantes hasta el final y se compararon **en el reparto**: comparten el 87 % de los alumnos con plata y ninguna combinación pierde más de 3 % contra la mejor. Como no cambia la decisión, gana la versión más simple.

### La entrega

Como en `entrega_ejemplo.csv`, van solo los alumnos que reciben plata. Los montos se redondean **hacia abajo** al centavo, para que la suma nunca pase el millón por redondeo.

In [23]:
def armar_entrega(m):
    e = pd.DataFrame({"id_alumno": foto["id_alumno"].to_numpy(), "monto": np.floor(m * 100) / 100})
    return e[e["monto"] > 0].sort_values("monto", ascending=False).reset_index(drop=True)


def validar(e):
    foto_2021 = d_raw.loc[d_raw["agno"] == 2021]
    egresados = set(foto_2021.loc[foto_2021["egresa"] == 1, "id_alumno"])
    assert list(e.columns) == ["id_alumno", "monto"]
    assert e["id_alumno"].is_unique
    assert e["id_alumno"].isin(foto_2021["id_alumno"]).all()
    assert not e["id_alumno"].isin(egresados).any()
    assert e["monto"].notna().all() and (e["monto"] >= 0).all()
    assert e["monto"].sum() <= PRESUPUESTO
    return {"alumnos": len(e), "suma": round(e["monto"].sum(), 2), "mínimo": e["monto"].min(),
            "mediana": e["monto"].median(), "máximo": e["monto"].max()}


entrega = armar_entrega(repartos[("sin E", "normal")])
entrega.to_csv("entrega.csv", index=False)
validar(entrega)

{'alumnos': 6419,
 'suma': np.float64(999967.94),
 'mínimo': np.float64(0.03),
 'mediana': np.float64(124.06),
 'máximo': np.float64(431.87)}

### Anexo: el reparto "como 2020"

No es una segunda entrega: la oficial es `entrega.csv`. Lo adjuntamos porque, cuando se corrija con lo que pasó en 2022, se pueden puntuar los dos repartos y saber **qué mundo ocurrió**. Es la única forma de cerrar la pregunta de los escenarios. Va en una carpeta aparte para que no se confunda con la entrega.

In [24]:
Path("anexos").mkdir(exist_ok=True)
anexo = armar_entrega(repartos[("sin E", "como 2020")])
anexo.to_csv("anexos/reparto_como_2020.csv", index=False)
validar(anexo)

{'alumnos': 8815,
 'suma': np.float64(999956.34),
 'mínimo': np.float64(14.71),
 'mediana': np.float64(67.2),
 'máximo': np.float64(545.36)}

Validaciones de la consigna: una fila por alumno, solo alumnos de la foto 2021, ningún egresado, montos ≥ 0 y suma ≤ US$ 1.000.000.

**Lo que esperamos:** unos 688 abandonos evitados *si* el modelo está bien calibrado para 2022 *y* el efecto de la plata es el que supone el ministerio. Lo primero lo discutimos en la sección 8. Lo segundo es un supuesto causal: predecir quién abandona no dice qué pasa si intervenimos.